# Priorización automática de baches a partir de fotos ciudadanas
**Inteligencia Artificial Avanzada · Proyecto integrador (Momento 3) · Ruta A · Tema libre**
Andrés Nicolás Sepúlveda Amaya · Fundación Universitaria Compensar

El notebook corre de principio a fin: descarga los datos, hace el EDA, separa train/val/test, entrena la línea base y dos redes profundas, evalúa, explica con Grad-CAM, analiza errores y genera los reportes en lenguaje natural.

**Antes de correr:** Entorno de ejecución → Cambiar tipo de entorno → **GPU T4**. Después: Entorno de ejecución → Ejecutar todo.

In [ ]:
import os, sys, json, glob, random, zipfile, urllib.request, time, datetime
import xml.etree.ElementTree as ET
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import cv2
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import f1_score, recall_score, accuracy_score, confusion_matrix, classification_report
from skimage.feature import hog

SEED = 42
random.seed(SEED); np.random.seed(SEED); tf.random.set_seed(SEED)

SMOKE = os.environ.get("SMOKE") == "1"          # prueba rapida con datos sinteticos
PAISES = ["India"]                               # se puede agregar "Japan", "United_States"
IMG = 160 if not SMOKE else 64
BATCH = 32
EP_CNN, EP_TL1, EP_TL2 = (12, 6, 6) if not SMOKE else (1, 1, 1)
DATA = "datos"; OUT = "resultados"
os.makedirs(DATA, exist_ok=True); os.makedirs(OUT, exist_ok=True)
CLASES = ["baja", "media", "alta"]
R = {"fecha_ejecucion": datetime.datetime.now().isoformat(timespec="seconds"), "semilla": SEED, "paises": PAISES, "img": IMG}
print("TF", tf.__version__, "| GPU:", tf.config.list_physical_devices("GPU"))

## 1. Datos
**Fuente:** RDD2022 (Arya et al.), publicado en figshare y en el repositorio oficial `sekilab/RoadDamageDetector`. Se descarga el subconjunto por país desde el enlace oficial. La fecha de descarga queda guardada en `resultados/resultados.json`.

In [ ]:
URL = "https://bigdatacup.s3.ap-northeast-1.amazonaws.com/2022/CRDDC2022/RDD2022/Country_Specific_Data_CRDDC2022/RDD2022_{}.zip"
if not SMOKE:
    for p in PAISES:
        z = f"{DATA}/RDD2022_{p}.zip"
        if not os.path.exists(z):
            print("Descargando", p, "..."); urllib.request.urlretrieve(URL.format(p), z)
        if not os.path.isdir(f"{DATA}/{p}"):
            with zipfile.ZipFile(z) as f: f.extractall(DATA)
        R.setdefault("tam_zip_MB", {})[p] = round(os.path.getsize(z)/1e6, 1)
R["fecha_descarga"] = datetime.date.today().isoformat()
print(os.listdir(DATA))

In [ ]:
# Lectura de las anotaciones (formato PASCAL VOC): una fila por imagen
filas = []
for x in glob.glob(f"{DATA}/**/*.xml", recursive=True):
    try: root = ET.parse(x).getroot()
    except Exception: continue
    stem = os.path.splitext(os.path.basename(x))[0]
    base = x.split("annotations")[0]
    img = None
    for ext in (".jpg", ".JPG", ".jpeg", ".png"):
        c = os.path.join(base, "images", stem + ext)
        if os.path.exists(c): img = c; break
    if img is None: continue
    sz = root.find("size"); W = float(sz.find("width").text); H = float(sz.find("height").text)
    objs = []
    for o in root.findall("object"):
        b = o.find("bndbox")
        x1, y1, x2, y2 = [float(b.find(k).text) for k in ("xmin", "ymin", "xmax", "ymax")]
        objs.append((o.find("name").text, x1, y1, x2, y2))
    pais = [p for p in x.replace("\\", "/").split("/") if p in PAISES + ["Sintetico"]]
    filas.append(dict(img=img, pais=pais[0] if pais else "?", W=W, H=H, objs=objs))
df = pd.DataFrame(filas)
print("Imagenes con anotacion:", len(df))

## 2. Análisis exploratorio (EDA)

In [ ]:
NOMBRES = {"D00": "Grieta longitudinal", "D10": "Grieta transversal", "D20": "Piel de cocodrilo", "D40": "Bache"}
df["n_obj"] = df.objs.apply(len)
for k in NOMBRES: df["n_" + k] = df.objs.apply(lambda o: sum(1 for a in o if a[0] == k))
eda = {"imagenes": int(len(df)), "sin_danio": int((df.n_obj == 0).sum()),
       "instancias_por_tipo": {NOMBRES[k]: int(df["n_" + k].sum()) for k in NOMBRES},
       "imagenes_con_bache": int((df.n_D40 > 0).sum()),
       "resoluciones": df.groupby(["W", "H"]).size().sort_values(ascending=False).head(3).to_dict().__str__()}
R["eda"] = eda; print(json.dumps(eda, indent=1, ensure_ascii=False))
fig, ax = plt.subplots(1, 2, figsize=(11, 3.5))
ax[0].bar(list(eda["instancias_por_tipo"].keys()), list(eda["instancias_por_tipo"].values()), color="#444")
ax[0].set_title("Instancias por tipo de daño"); ax[0].tick_params(axis="x", rotation=20)
ax[1].hist(df.n_obj, bins=range(0, 12), color="#444"); ax[1].set_title("Daños por imagen"); ax[1].set_xlabel("cantidad")
plt.tight_layout(); plt.savefig(f"{OUT}/fig1_eda.png", dpi=150); plt.show()

## 3. Etiqueta de severidad (etiqueta débil)
RDD2022 marca **dónde** está el bache, no **qué tan grave** es. La severidad se calcula con una regla escrita antes de entrenar:

`puntaje = área del bache más grande / área de la imagen`, y se suma un 25 % por cada bache adicional (máximo 3).

Los cortes entre baja, media y alta son los terciles del puntaje **calculados solo con el conjunto de entrenamiento**, para no filtrar información de la prueba. Limitación conocida: el área depende de qué tan cerca se tomó la foto.

In [ ]:
def puntaje(objs, W, H):
    a = sorted([(x2 - x1) * (y2 - y1) / (W * H) for (c, x1, y1, x2, y2) in objs if c == "D40"], reverse=True)
    if not a: return np.nan
    return a[0] * (1 + 0.25 * min(len(a) - 1, 3))
df["score"] = [puntaje(o, w, h) for o, w, h in zip(df.objs, df.W, df.H)]
b = df[df.score.notna()].reset_index(drop=True)
print("Imagenes con al menos un bache:", len(b))

# Division ANTES de cualquier transformacion (70 / 15 / 15), semilla fija
tr, tmp = train_test_split(b, test_size=0.30, random_state=SEED)
va, te = train_test_split(tmp, test_size=0.50, random_state=SEED)
q1, q2 = tr.score.quantile([1/3, 2/3]).values           # cortes SOLO con train
def sev(s): return 0 if s <= q1 else (1 if s <= q2 else 2)
for d in (tr, va, te): d["y"] = d.score.apply(sev)
R["cortes_severidad"] = {"baja_media": float(q1), "media_alta": float(q2)}
R["division"] = {n: {"n": int(len(d)), "por_clase": {CLASES[i]: int((d.y == i).sum()) for i in range(3)}} for n, d in (("train", tr), ("val", va), ("test", te))}
print(json.dumps(R["division"], indent=1)); print("Cortes:", q1, q2)

fig, ax = plt.subplots(1, 3, figsize=(11, 3.6))
for i in range(3):
    r = tr[tr.y == i].iloc[0]; ax[i].imshow(cv2.cvtColor(cv2.imread(r.img), cv2.COLOR_BGR2RGB)); ax[i].set_title("Severidad " + CLASES[i]); ax[i].axis("off")
plt.tight_layout(); plt.savefig(f"{OUT}/fig2_ejemplos.png", dpi=150); plt.show()

### Etiquetas manuales (opcional pero recomendado)
La celda exporta una muestra del conjunto de prueba a `resultados/para_etiquetar/` con un CSV. Si existe `etiquetas_manuales.csv` (columnas `archivo,severidad` con baja/media/alta) y la carpeta `fotos_propias/` con su CSV, el notebook también evalúa sobre ellas.

In [ ]:
import shutil
os.makedirs(f"{OUT}/para_etiquetar", exist_ok=True)
m = te.sample(min(300, len(te)), random_state=SEED)
for p in m.img: shutil.copy(p, f"{OUT}/para_etiquetar/" + os.path.basename(p))
pd.DataFrame({"archivo": [os.path.basename(p) for p in m.img], "severidad": ""}).to_csv(f"{OUT}/para_etiquetar/plantilla_etiquetas.csv", index=False)
print("Muestra para etiquetar a mano:", len(m))

## 4. Línea base: HOG + histograma de color + regresión logística

In [ ]:
def leer(p, s=IMG):
    im = cv2.imread(p); im = cv2.cvtColor(im, cv2.COLOR_BGR2RGB); return cv2.resize(im, (s, s))
def feats(p):
    im = leer(p, 128); g = cv2.cvtColor(im, cv2.COLOR_RGB2GRAY)
    h = hog(g, orientations=9, pixels_per_cell=(16, 16), cells_per_block=(2, 2))
    c = np.concatenate([np.histogram(im[..., k], bins=16, range=(0, 255))[0] for k in range(3)]) / (128 * 128)
    return np.concatenate([h, c])
t0 = time.time()
Xtr = np.array([feats(p) for p in tr.img]); Xte = np.array([feats(p) for p in te.img])
base = make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000, C=0.1, random_state=SEED))  # el escalador se ajusta solo con train
base.fit(Xtr, tr.y)
pred = {"Linea base (HOG + RL)": base.predict(Xte)}
print("Listo en", round(time.time() - t0), "s")

## 5. Redes profundas

In [ ]:
def ds(d, train=False):
    x = tf.data.Dataset.from_tensor_slices((d.img.values, d.y.values.astype("int32")))
    def f(p, y):
        im = tf.io.decode_image(tf.io.read_file(p), channels=3, expand_animations=False)
        return tf.image.resize(im, (IMG, IMG)), y
    if train: x = x.shuffle(len(d), seed=SEED)
    return x.map(f, num_parallel_calls=tf.data.AUTOTUNE).batch(BATCH).prefetch(tf.data.AUTOTUNE)
dtr, dva, dte = ds(tr, True), ds(va), ds(te)
aug = keras.Sequential([layers.RandomFlip("horizontal"), layers.RandomBrightness(0.15), layers.RandomContrast(0.15)], name="aumento")
cb = lambda: [keras.callbacks.EarlyStopping(monitor="val_loss", patience=3, restore_best_weights=True)]
hist = {}

# --- Modelo 1: CNN propia desde cero ---
def bloque(x, f):
    x = layers.Conv2D(f, 3, padding="same", activation="relu")(x); x = layers.BatchNormalization()(x); return layers.MaxPooling2D()(x)
i = keras.Input((IMG, IMG, 3)); x = aug(i); x = layers.Rescaling(1 / 255.)(x)
for f in (32, 64, 128, 256): x = bloque(x, f)
x = layers.GlobalAveragePooling2D()(x); x = layers.Dropout(0.4)(x)
cnn = keras.Model(i, layers.Dense(3, activation="softmax")(x), name="cnn_propia")
cnn.compile(keras.optimizers.Adam(1e-3), "sparse_categorical_crossentropy", metrics=["accuracy"])
hist["CNN propia"] = cnn.fit(dtr, validation_data=dva, epochs=EP_CNN, callbacks=cb(), verbose=2).history
pred["CNN propia"] = cnn.predict(dte, verbose=0).argmax(1)

In [ ]:
# --- Modelo 2: MobileNetV2 con transfer learning (2 fases) ---
try:
    mb = keras.applications.MobileNetV2(input_shape=(IMG, IMG, 3), include_top=False, weights=None if SMOKE else "imagenet")
except Exception as e:
    print("Sin pesos de ImageNet:", e); mb = keras.applications.MobileNetV2(input_shape=(IMG, IMG, 3), include_top=False, weights=None)
mb.trainable = False
i = keras.Input((IMG, IMG, 3)); x = aug(i); x = layers.Rescaling(1 / 127.5, offset=-1)(x)
x = mb(x, training=False); x = layers.GlobalAveragePooling2D()(x); x = layers.Dropout(0.3)(x)
tl = keras.Model(i, layers.Dense(3, activation="softmax")(x), name="mobilenetv2_tl")
tl.compile(keras.optimizers.Adam(1e-3), "sparse_categorical_crossentropy", metrics=["accuracy"])
h1 = tl.fit(dtr, validation_data=dva, epochs=EP_TL1, callbacks=cb(), verbose=2).history       # fase 1: solo la cabeza
mb.trainable = True
for l in mb.layers[:-30]: l.trainable = False                                                  # fase 2: ajuste fino de las ultimas capas
tl.compile(keras.optimizers.Adam(1e-5), "sparse_categorical_crossentropy", metrics=["accuracy"])
h2 = tl.fit(dtr, validation_data=dva, epochs=EP_TL2, callbacks=cb(), verbose=2).history
hist["MobileNetV2 (transfer learning)"] = {k: h1[k] + h2[k] for k in h1}
pred["MobileNetV2 (transfer learning)"] = tl.predict(dte, verbose=0).argmax(1)

fig, ax = plt.subplots(1, 2, figsize=(11, 3.5))
for n, h in hist.items():
    ax[0].plot(h["loss"], label=n + " train"); ax[0].plot(h["val_loss"], "--", label=n + " val"); ax[1].plot(h["val_accuracy"], label=n)
ax[0].set_title("Pérdida"); ax[1].set_title("Exactitud en validación"); ax[0].legend(fontsize=7); ax[1].legend(fontsize=7)
plt.tight_layout(); plt.savefig(f"{OUT}/fig3_curvas.png", dpi=150); plt.show()
R["epocas"] = {n: len(h["loss"]) for n, h in hist.items()}

## 6. Resultados en el conjunto de prueba
Métrica principal: **recall de la clase alta** (el error más caro es dejar un bache grave al final de la cola) y **F1-macro**. Se agrega una referencia de "clase mayoritaria" para ver si los modelos aprenden algo.

In [ ]:
yte = te.y.values
pred["Clase mayoritaria"] = np.full_like(yte, np.bincount(tr.y).argmax())
def met(y, p): return {"recall_alta": round(float(recall_score(y, p, labels=[2], average="macro", zero_division=0)), 3),
                       "f1_macro": round(float(f1_score(y, p, average="macro", zero_division=0)), 3),
                       "exactitud": round(float(accuracy_score(y, p)), 3)}
orden = ["Clase mayoritaria", "Linea base (HOG + RL)", "CNN propia", "MobileNetV2 (transfer learning)"]
R["test"] = {n: met(yte, pred[n]) for n in orden}
R["matrices"] = {n: confusion_matrix(yte, pred[n], labels=[0, 1, 2]).tolist() for n in orden}
tabla = pd.DataFrame(R["test"]).T; print(tabla)
MEJOR = max(["CNN propia", "MobileNetV2 (transfer learning)"], key=lambda n: (R["test"][n]["f1_macro"], R["test"][n]["recall_alta"]))
R["mejor_modelo"] = MEJOR; modelo = cnn if MEJOR == "CNN propia" else tl
print("\nMejor modelo profundo:", MEJOR); print(classification_report(yte, pred[MEJOR], target_names=CLASES, zero_division=0))
fig, ax = plt.subplots(1, 3, figsize=(12, 3.6))
for a, n in zip(ax, orden[1:]):
    cm = np.array(R["matrices"][n]); a.imshow(cm, cmap="Greys"); a.set_title(n, fontsize=9)
    a.set_xticks(range(3)); a.set_yticks(range(3)); a.set_xticklabels(CLASES); a.set_yticklabels(CLASES); a.set_xlabel("Predicho"); a.set_ylabel("Real")
    for r_ in range(3):
        for c_ in range(3): a.text(c_, r_, cm[r_, c_], ha="center", va="center", color="white" if cm[r_, c_] > cm.max() / 2 else "black")
plt.tight_layout(); plt.savefig(f"{OUT}/fig4_matrices.png", dpi=150); plt.show()

In [ ]:
# Evaluacion con etiquetas manuales y fotos propias, si existen
def evalua_csv(csv, carpeta, nombre):
    if not os.path.exists(csv): print("No hay", csv, "-> se omite"); return
    e = pd.read_csv(csv).dropna(); e = e[e.severidad.isin(CLASES)]
    X = np.array([leer(os.path.join(carpeta, a)) for a in e.archivo]).astype("float32")
    p = modelo.predict(X, verbose=0).argmax(1); y = e.severidad.map({c: i for i, c in enumerate(CLASES)}).values
    R[nombre] = dict(met(y, p), n=int(len(e)), matriz=confusion_matrix(y, p, labels=[0, 1, 2]).tolist()); print(nombre, R[nombre])
evalua_csv("etiquetas_manuales.csv", f"{OUT}/para_etiquetar", "test_manual")
evalua_csv("fotos_propias/etiquetas.csv", "fotos_propias", "prueba_campo")

## 7. Explicabilidad: Grad-CAM

In [ ]:
def ultima_conv(m):
    for l in reversed(m.layers):
        if isinstance(l, layers.Conv2D): return m, l.name
    return None, None
def gradcam(im):
    x = tf.convert_to_tensor(im[None].astype("float32"))
    if modelo.name == "cnn_propia":
        _, capa = ultima_conv(modelo); g = keras.Model(modelo.input, [modelo.get_layer(capa).output, modelo.output])
        with tf.GradientTape() as t:
            fm, p = g(x, training=False); k = tf.argmax(p[0]); s = p[:, k]
    else:
        with tf.GradientTape() as t:
            z = layers.Rescaling(1 / 127.5, offset=-1)(x); fm = mb(z, training=False); t.watch(fm)
            p = modelo.layers[-1](modelo.layers[-3](fm)); k = tf.argmax(p[0]); s = p[:, k]
    gr = t.gradient(s, fm); w = tf.reduce_mean(gr, axis=(1, 2))
    cam = tf.nn.relu(tf.reduce_sum(fm * w[:, None, None, :], -1))[0].numpy()
    cam = cv2.resize(cam / (cam.max() + 1e-8), (IMG, IMG)); return cam, int(k), float(p[0, k])
def superpone(im, cam):
    h = cv2.applyColorMap(np.uint8(255 * cam), cv2.COLORMAP_JET)[..., ::-1]; return np.uint8(0.55 * im + 0.45 * h)
fig, ax = plt.subplots(2, 4, figsize=(12, 6))
for j, (_, r) in enumerate(te.sample(4, random_state=SEED).iterrows()):
    im = leer(r.img); cam, k, pr = gradcam(im)
    ax[0, j].imshow(im); ax[0, j].set_title(f"real: {CLASES[r.y]}", fontsize=9); ax[0, j].axis("off")
    ax[1, j].imshow(superpone(im, cam)); ax[1, j].set_title(f"pred: {CLASES[k]} ({pr:.2f})", fontsize=9); ax[1, j].axis("off")
plt.tight_layout(); plt.savefig(f"{OUT}/fig5_gradcam.png", dpi=150); plt.show()

## 8. Análisis de errores (10 casos fallidos)

In [ ]:
te2 = te.copy(); te2["pred"] = pred[MEJOR]; mal = te2[te2.pred != te2.y].copy()
mal["dist"] = (mal.pred - mal.y).abs(); mal = mal.sort_values(["dist", "score"], ascending=[False, False]).head(10)
err = []
fig, ax = plt.subplots(2, 5, figsize=(14, 6)); ax = ax.ravel()
for j, (_, r) in enumerate(mal.iterrows()):
    im = leer(r.img); cam, _, _ = gradcam(im); bri = float(cv2.cvtColor(im, cv2.COLOR_RGB2GRAY).mean())
    ax[j].imshow(superpone(im, cam)); ax[j].set_title(f"#{j+1} real {CLASES[r.y]} / pred {CLASES[r.pred]}", fontsize=8); ax[j].axis("off")
    err.append({"caso": j + 1, "archivo": os.path.basename(r.img), "real": CLASES[r.y], "pred": CLASES[r.pred], "puntaje_area": round(float(r.score), 4),
                "n_baches": int(r.n_D40), "otros_danios": int(r.n_obj - r.n_D40), "brillo_medio": round(bri, 1),
                "cerca_del_corte": bool(min(abs(r.score - q1), abs(r.score - q2)) < 0.15 * (q2 - q1 + 1e-9))})
for a in ax[len(mal):]: a.axis("off")
plt.tight_layout(); plt.savefig(f"{OUT}/fig6_errores.png", dpi=150); plt.show()
R["errores"] = err; R["tipo_error"] = {"subestima (pred < real)": int((te2.pred < te2.y).sum()), "sobreestima (pred > real)": int((te2.pred > te2.y).sum()), "alta_marcada_como_baja": int(((te2.y == 2) & (te2.pred == 0)).sum())}
pd.DataFrame(err)

## 9. Riesgo y reporte en lenguaje natural (componente NLP)
El riesgo de la plataforma combina la severidad visual con la posición que marca el ciudadano. El reporte para la entidad lo redacta un modelo de lenguaje (Transformer) a partir de una plantilla con los datos; si el modelo no carga, queda la plantilla. Se evalúan 10 reportes con una lista de chequeo.

In [ ]:
REGLA = {("baja", "borde"): "Baja", ("baja", "carril"): "Media", ("media", "borde"): "Media", ("media", "carril"): "Alta", ("alta", "borde"): "Alta", ("alta", "carril"): "Crítico"}
def riesgo(sev, pos): return REGLA[(sev, pos)]
def plantilla(d):
    return (f"Reporte de daño vial. Tipo de daño: bache. Cantidad de baches detectados en la foto: {d['n']}. Severidad visual estimada por el modelo: {d['sev']} "
            f"(confianza {d['conf']:.0%}). Posición marcada por el ciudadano: {'carril de circulación' if d['pos'] == 'carril' else 'borde o berma'}. "
            f"Riesgo sugerido: {d['riesgo']}. La prioridad es una sugerencia automática y debe ser validada por una persona.")
gen = None
if not SMOKE:
    try:
        from transformers import pipeline
        gen = pipeline("text-generation", model="Qwen/Qwen2.5-0.5B-Instruct", device_map="auto")
    except Exception as e: print("No se pudo cargar el modelo de lenguaje:", e)
def redacta(d):
    base_txt = plantilla(d)
    if gen is None: return base_txt, "plantilla"
    msg = [{"role": "system", "content": "Eres un asistente que redacta reportes de daños viales para una entidad de mantenimiento vial en Colombia. Escribe en español, en un solo párrafo de máximo 90 palabras, tono formal y claro. Usa únicamente los datos entregados. No inventes direcciones, medidas ni fechas."},
           {"role": "user", "content": "Redacta el reporte con estos datos: " + base_txt}]
    try:
        o = gen(msg, max_new_tokens=160, do_sample=False)[0]["generated_text"][-1]["content"].strip(); return o, "modelo de lenguaje"
    except Exception as e: return base_txt, "plantilla"
def chequeo(txt, d):
    t = txt.lower()
    return {"menciona_severidad": d["sev"] in t, "menciona_riesgo": d["riesgo"].lower() in t, "menciona_posicion": ("carril" in t) if d["pos"] == "carril" else ("borde" in t or "berma" in t),
            "aclara_validacion_humana": any(w in t for w in ("validad", "validar", "persona", "revis", "verific")), "longitud_ok": len(txt.split()) <= 120}
reps = []
for j, (_, r) in enumerate(te.sample(10, random_state=7).iterrows()):
    im = leer(r.img); _, k, pr = gradcam(im); pos = "carril" if j % 2 == 0 else "borde"     # la posicion la marca el ciudadano; aqui se simula
    d = dict(n=int(r.n_D40), sev=CLASES[k], conf=pr, pos=pos); d["riesgo"] = riesgo(d["sev"], pos)
    txt, origen = redacta(d); ch = chequeo(txt, d); reps.append({"archivo": os.path.basename(r.img), **d, "origen": origen, "reporte": txt, "chequeo": ch, "puntos": int(sum(ch.values()))})
R["reportes"] = reps; R["reportes_resumen"] = {"origen": reps[0]["origen"], "promedio_puntos_de_5": round(float(np.mean([x["puntos"] for x in reps])), 2),
    "por_criterio": {k: int(sum(x["chequeo"][k] for x in reps)) for k in reps[0]["chequeo"]}}
print(json.dumps(R["reportes_resumen"], indent=1, ensure_ascii=False)); print("\nEjemplo:\n", reps[0]["reporte"])

## 10. Guardar todo

In [ ]:
modelo.save(f"{OUT}/modelo_severidad.keras")
json.dump({"clases": CLASES, "img": IMG, "modelo": MEJOR}, open(f"{OUT}/config.json", "w"))
json.dump(R, open(f"{OUT}/resultados.json", "w"), indent=1, ensure_ascii=False)
with zipfile.ZipFile("resultados.zip", "w", zipfile.ZIP_DEFLATED) as z:
    for f in glob.glob(f"{OUT}/*.png") + [f"{OUT}/resultados.json", f"{OUT}/config.json"]: z.write(f)
print("Listo. Archivo para el informe: resultados.zip")
try:
    from google.colab import files; files.download("resultados.zip")
except Exception: pass